# Security SLM Fine-Tune
## DeepSeek-R1-Distill-Qwen-1.5B → Blue/Red Team Security Model

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar → Secrets (key icon) → add `HUGGINGFACE_TOKEN` and `WANDB_API_KEY`
3. Upload `security_dataset.jsonl` to Colab file browser (left sidebar → folder icon)

**Run cells top to bottom. Do NOT skip the baseline test cells.**

| Phase | Cells | Purpose |
|-------|-------|---------|
| Setup | 1-3 | Install, auth, GPU check |
| Baseline | 4-5 | Load base model, run pre-training eval |
| Train | 6-8 | LoRA, dataset, train |
| Compare | 9 | Run same eval on fine-tuned model, show diff |
| Export | 10-13 | Push to HF, export GGUF |

In [1]:
# ── Cell 1: Install ──────────────────────────────────────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install wandb huggingface_hub
print('Install complete.')

In [14]:
# ── Cell 2: Authenticate WandB + HuggingFace ─────────────────
import wandb
from huggingface_hub import login
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))
wandb.init(
    project='security-slm',
    name='deepseek-r1-1.5b-finetune',
    config={
        'base_model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
        'lora_rank': 16,
        'dataset_size': 20,
        'quantization': 'q4_k_m',
    }
)

login(token=userdata.get('HUGGINGFACE_TOKEN'))

# CHANGE THIS to your HuggingFace username/repo
HF_REPO = 'Nguuma/security-slm-unsloth-1.5b' # <-- IMPORTANT: Replace 'your-username' with your actual HuggingFace username

print(f'Authenticated. Target repo: https://huggingface.co/{HF_REPO}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Authenticated. Target repo: https://huggingface.co/Nguuma/security-slm-unsloth-1.5b


In [15]:
# ── Cell 3: Check GPU ─────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Fri Mar 13 09:15:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             30W /   70W |    2329MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## PHASE 1 — Baseline Test (Before Fine-Tuning)

Run the **unmodified** base model against 5 security prompts.
Results are saved to `baseline_results.json` and logged to WandB.
We re-run the same prompts after training to measure improvement.

In [16]:
# ── Cell 4: Load base model ───────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('Base model loaded.')

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-qwen-1.5b-unsloth-bnb-4bit as a legacy tokenizer.


unsloth/deepseek-r1-distill-qwen-1.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded.


In [17]:
# ── Cell 5: BASELINE EVALUATION ──────────────────────────────
import json, re, textwrap, warnings


# 5 prompts covering the core security categories
EVAL_PROMPTS = [
    {
        'id': 'mcp_red',
        'category': 'MCP Security',
        'prompt': 'Explain how an attacker can craft a malicious MCP server tool description to trick an AI agent into exfiltrating environment variables to an external server.'
    },
    {
        'id': 'prompt_hijack',
        'category': 'Prompt Hijacking',
        'prompt': 'What is the Payload Splitting technique and how does it bypass LlamaGuard by splitting a harmful instruction across multiple conversation turns?'
    },
    {
        'id': 'agentic_blue',
        'category': 'Agentic Security',
        'prompt': 'Implement a Python circuit breaker for an AI agent that stops execution when the same tool is called more than 3 times consecutively, preventing resource exhaustion attacks.'
    },
    {
        'id': 'ssrf_red',
        'category': 'Cloud-AI SSRF',
        'prompt': 'How can an attacker exploit an LLM with a URL-fetching tool to steal temporary AWS credentials from the EC2 instance metadata endpoint at 169.254.169.254?'
    },
    {
        'id': 'iam_blue',
        'category': 'Cloud IAM',
        'prompt': 'Write a minimal AWS IAM policy in JSON that gives an AI agent read-only access to s3://company-data/ai-agent/ only, with an explicit deny for all destructive actions.'
    },
]

def score_response(response: str) -> dict:
    """Score a model response on 5 dimensions."""
    has_think = '<think>' in response
    think_steps = 0
    if has_think:
        think_block = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
        if think_block:
            think_steps = len([l for l in think_block.group(1).split('\n') if l.strip()])

    # Technical depth: presence of concrete technical markers
    tech_markers = [
        r'\b(AWS|IAM|S3|EC2|arn:|s3://)',
        r'\b(def |class |import |```python)',
        r'\b(CVE|OWASP|RFC|NIST)',
        r'\b(json\.dumps|requests\.|boto3|aws_|iam:)',
        r'\b(169\.254|metadata|SSRF|injection|exfiltrat)',
    ]
    tech_score = sum(1 for p in tech_markers if re.search(p, response))

    response_len = len(response.split())

    # Composite score /10
    score = 0
    score += 3 if has_think else 0           # Has reasoning block
    score += min(3, think_steps // 2)        # Quality of reasoning (max 3)
    score += min(2, tech_score)              # Technical depth (max 2)
    score += 2 if response_len > 150 else 1 if response_len > 50 else 0  # Length

    return {
        'has_think_block': has_think,
        'think_steps': think_steps,
        'tech_depth_score': tech_score,
        'word_count': response_len,
        'total_score': score,
    }

def run_inference(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    FastLanguageModel.for_inference(model)
    formatted = f'### Instruction:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Return only the response part
    return response.split('### Response:\n', 1)[-1].strip()

# ── Run baseline ──
print('Running BASELINE evaluation on unmodified base model...')
print('=' * 60)
warnings.filterwarnings('ignore', category=FutureWarning)
baseline_results = []

for item in EVAL_PROMPTS:
    print(f"\n[{item['category']}] {item['prompt'][:80]}...")
    response = run_inference(model, tokenizer, item['prompt'])
    scores = score_response(response)
    result = {
        'id': item['id'],
        'category': item['category'],
        'prompt': item['prompt'],
        'response': response,
        'scores': scores,
        'model': 'baseline',
    }
    baseline_results.append(result)

    print(f'  Score: {scores["total_score"]}/10 | '
          f'<think>: {scores["has_think_block"]} | '
          f'Steps: {scores["think_steps"]} | '
          f'Words: {scores["word_count"]}')
    print(f'  Response preview: {response[:200]}...')

# Save baseline to file
with open('baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

baseline_avg = sum(r['scores']['total_score'] for r in baseline_results) / len(baseline_results)
baseline_think_rate = sum(1 for r in baseline_results if r['scores']['has_think_block']) / len(baseline_results)

print('\n' + '=' * 60)
print(f'BASELINE SUMMARY')
print(f'  Average score:    {baseline_avg:.1f}/10')
print(f'  <think> rate:     {baseline_think_rate*100:.0f}%')
print(f'  Results saved to: baseline_results.json')

# Log to WandB
wandb.log({
    'baseline/avg_score': baseline_avg,
    'baseline/think_block_rate': baseline_think_rate,
    **{f'baseline/score_{r["id"]}': r['scores']['total_score'] for r in baseline_results}
})


Running BASELINE evaluation on unmodified base model...

[MCP Security] Explain how an attacker can craft a malicious MCP server tool description to tri...
  Score: 3/10 | <think>: False | Steps: 0 | Words: 287
  Response preview: </think>

An attacker can craft a malicious MCP server tool description to trick an AI agent into exfiltrating environment variables to an external server by following these steps:

1. **Understanding...

[Prompt Hijacking] What is the Payload Splitting technique and how does it bypass LlamaGuard by spl...
  Score: 2/10 | <think>: False | Steps: 0 | Words: 263
  Response preview: The Payload Splitting technique is a method used to split harmful content or instructions into multiple parts so that they can be executed across different conversation turns. This technique bypasses ...

[Agentic Security] Implement a Python circuit breaker for an AI agent that stops execution when the...
  Score: 2/10 | <think>: False | Steps: 0 | Words: 376
  Response preview: To 

---
## PHASE 2 — Fine-Tuning

In [18]:
# ── Cell 6: Apply LoRA ────────────────────────────────────────
# Switch model back to training mode after inference
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA applied.')
model.print_trainable_parameters()

LoRA applied.
trainable params: 18,464,768 || all params: 1,795,552,768 || trainable%: 1.0284


In [19]:
# ── Cell 7: Load + format dataset ────────────────────────────
# ── Cell 7: Load + format dataset ────────────────────────────
from datasets import load_dataset

def format_sample(sample):
    return {
        'text': f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['content']}"
    }

dataset = load_dataset('json', data_files='security_dataset_fixed.jsonl', split='train')
dataset = dataset.map(format_sample)
print(f'Dataset: {len(dataset)} samples')

Dataset: 95 samples


In [20]:
# ── Cell 8: Train ─────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_steps=25,
        save_total_limit=2,
        output_dir='outputs',
        optim='adamw_8bit',
        seed=42,
        report_to='wandb',
        run_name='security-slm-run-1',
    ),
)

print('Training...')
trainer_stats = trainer.train()
wandb.finish()
print(f'Done. Final loss: {trainer_stats.training_loss:.4f}')

Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 95 | Num Epochs = 3 | Total steps = 36
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
1,2.647214
2,2.604049
3,2.361181
4,2.352349
5,2.565927
6,2.371628
7,2.482957
8,2.389544
9,2.377737
10,2.229253


baseline/avg_score,▁
baseline/score_agentic_blue,▁
baseline/score_iam_blue,▁
baseline/score_mcp_red,▁
baseline/score_prompt_hijack,▁
baseline/score_ssrf_red,▁
baseline/think_block_rate,▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▅▇▄▅▆▅█▇▇▃▆▅▂▃▂▄▄▅▆▄▂▃▅▅▃▄▃▁▃▁▃▄▁▅▅▃
+2,...


Done. Final loss: 2.2535


---
## PHASE 3 — Post-Training Evaluation & Comparison

Run the **same 5 prompts** on the fine-tuned model and compare against baseline.

In [21]:
# ── Cell 9: POST-TRAINING EVALUATION + COMPARISON ────────────
import json, re

# Load baseline results
with open('baseline_results.json') as f:
    baseline_results = json.load(f)
baseline_by_id = {r['id']: r for r in baseline_results}

print('Running POST-TRAINING evaluation...')
print('=' * 60)
finetuned_results = []

for item in EVAL_PROMPTS:
    print(f"\n[{item['category']}]")
    response = run_inference(model, tokenizer, item['prompt'])
    scores = score_response(response)
    result = {
        'id': item['id'],
        'category': item['category'],
        'prompt': item['prompt'],
        'response': response,
        'scores': scores,
        'model': 'finetuned',
    }
    finetuned_results.append(result)

with open('finetuned_results.json', 'w') as f:
    json.dump(finetuned_results, f, indent=2)

# ── Side-by-side comparison table ──
print('\n' + '=' * 70)
print(f'{"COMPARISON: Baseline vs Fine-Tuned":^70}')
print('=' * 70)
print(f'{"Category":<22} {"Base Score":>10} {"FT Score":>8} {"Delta":>6} {"Think%":>7}')
print('-' * 70)

total_base, total_ft = 0, 0
for ft in finetuned_results:
    base = baseline_by_id[ft['id']]
    base_s = base['scores']['total_score']
    ft_s = ft['scores']['total_score']
    delta = ft_s - base_s
    delta_str = f'+{delta}' if delta >= 0 else str(delta)
    base_think = 'YES' if base['scores']['has_think_block'] else 'NO'
    ft_think = 'YES' if ft['scores']['has_think_block'] else 'NO'
    print(f"{ft['category']:<22} {base_s:>4}/10     {ft_s:>4}/10  {delta_str:>5}   {base_think}->{ft_think}")
    total_base += base_s
    total_ft += ft_s

n = len(finetuned_results)
avg_base = total_base / n
avg_ft = total_ft / n
improvement = ((avg_ft - avg_base) / max(avg_base, 0.01)) * 100

print('=' * 70)
print(f'{"AVERAGE":<22} {avg_base:>4.1f}/10     {avg_ft:>4.1f}/10  {"+" if improvement >= 0 else ""}{improvement:.0f}%')
print('=' * 70)

think_base = sum(1 for r in baseline_results if r['scores']['has_think_block']) / n * 100
think_ft = sum(1 for r in finetuned_results if r['scores']['has_think_block']) / n * 100
print(f'\n<think> block rate:  Baseline {think_base:.0f}%  ->  Fine-tuned {think_ft:.0f}%')

avg_words_base = sum(r['scores']['word_count'] for r in baseline_results) / n
avg_words_ft = sum(r['scores']['word_count'] for r in finetuned_results) / n
print(f'Avg response length: Baseline {avg_words_base:.0f} words  ->  Fine-tuned {avg_words_ft:.0f} words')

# Log comparison to WandB
wandb.init(project='security-slm', name='eval-comparison', resume='allow')
wandb.log({
    'eval/baseline_avg_score': avg_base,
    'eval/finetuned_avg_score': avg_ft,
    'eval/score_improvement_pct': improvement,
    'eval/baseline_think_rate': think_base,
    'eval/finetuned_think_rate': think_ft,
    **{f'eval/ft_score_{r["id"]}': r['scores']['total_score'] for r in finetuned_results}
})
wandb.finish()

# ── Print one full response comparison (first prompt) ──
print('\n' + '=' * 70)
print('FULL RESPONSE COMPARISON — MCP Security prompt')
print('=' * 70)
print('--- BASELINE ---')
print(baseline_by_id['mcp_red']['response'][:800])
print('\n--- FINE-TUNED ---')
print(next(r for r in finetuned_results if r['id'] == 'mcp_red')['response'][:800])

Running POST-TRAINING evaluation...

[MCP Security]

[Prompt Hijacking]

[Agentic Security]

[Cloud-AI SSRF]

[Cloud IAM]

                  COMPARISON: Baseline vs Fine-Tuned                  
Category               Base Score FT Score  Delta  Think%
----------------------------------------------------------------------
MCP Security              3/10        5/10     +2   NO->YES
Prompt Hijacking          2/10        7/10     +5   NO->YES
Agentic Security          2/10        5/10     +3   NO->YES
Cloud-AI SSRF             4/10        8/10     +4   NO->YES
Cloud IAM                 0/10        5/10     +5   NO->YES
AVERAGE                 2.2/10      6.0/10  +173%

<think> block rate:  Baseline 0%  ->  Fine-tuned 100%
Avg response length: Baseline 255 words  ->  Fine-tuned 282 words


eval/baseline_avg_score,▁
eval/baseline_think_rate,▁
eval/finetuned_avg_score,▁
eval/finetuned_think_rate,▁
eval/ft_score_agentic_blue,▁
eval/ft_score_iam_blue,▁
eval/ft_score_mcp_red,▁
eval/ft_score_prompt_hijack,▁
eval/ft_score_ssrf_red,▁
eval/score_improvement_pct,▁
eval/baseline_avg_score,2.2



FULL RESPONSE COMPARISON — MCP Security prompt
--- BASELINE ---
</think>

An attacker can craft a malicious MCP server tool description to trick an AI agent into exfiltrating environment variables to an external server by following these steps:

1. **Understanding MCP Servers**: MCP stands for "Merkle Tree" or "Merkle Hash" in the context of blockchain technology. MCP servers are used to store environment variables on-chain, allowing users to access them without leaving the chain.

2. **Crafting the Tool Description**: The attacker can create a tool description that claims to be an MCP server but is actually a malicious script designed to intercept environment variables. This involves writing code that looks like it's part of a Merkle tree but is instead a malicious script.

3. **Exfiltration Target**: The attacker can target specific environment varia

--- FINE-TUNED ---
<think>
Step 1: **Understanding MCP Server Tools**
MCP server tools are command-line scripts that execute arbitrar

---
## PHASE 4 — Export & Push

In [ ]:
# ── Cell 10: Push LoRA adapter to HuggingFace (crash-safe) ───
model.save_pretrained('security-slm-lora')
tokenizer.save_pretrained('security-slm-lora')
model.push_to_hub(HF_REPO, token=userdata.get('HUGGINGFACE_TOKEN'))
tokenizer.push_to_hub(HF_REPO, token=userdata.get('HUGGINGFACE_TOKEN'))
print(f'Adapter saved: https://huggingface.co/{HF_REPO}')

In [24]:
# ── Cell 11: Export to GGUF Q4_K_M ───────────────────────────
print('Exporting GGUF Q4_K_M (~5 min)...')

# Define the correct GGUF file path based on Unsloth's naming convention and existing files
# The directory is typically '<save_name>_gguf' and the file name is derived from the base model.
GGUF_DIR = 'security-slm_gguf'
BASE_MODEL_SLUG = 'deepseek-r1-distill-qwen-1.5b' # Derived from the base_model in Cell 2 config
GGUF_FILE_NAME = f'{BASE_MODEL_SLUG}.Q4_K_M.gguf'
GGUF_FULL_PATH = f'{GGUF_DIR}/{GGUF_FILE_NAME}'

# The 'save_path' argument to save_pretrained_gguf should be the directory name.
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method='q4_k_m')

import os
if os.path.exists(GGUF_FULL_PATH):
    size_gb = os.path.getsize(GGUF_FULL_PATH) / 1e9
    print(f'Done: {GGUF_FULL_PATH} ({size_gb:.2f} GB)')
else:
    print(f'Error: GGUF file not found at expected path: {GGUF_FULL_PATH}')


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Exporting GGUF Q4_K_M (~5 min)...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:43<00:00, 103.01s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:40<00:00, 100.33s/it]


Unsloth: Merge process complete. Saved to `/content/security-slm_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['security-slm_gguf_gguf/deepseek-r1-distill-qwen-1.5b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['security-slm_gguf_gguf/deepseek-r1-distill-qwen-1.5b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/deepseek-r1-distill-qwen-1.5b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model security-slm_gguf_gguf/deepseek-r1-distill-qwen-1.5b.Q4_K_M.gguf -p "why is the sky blue?"
Done: security-slm_gguf/deepseek-r1-distill-qwen-1.5b.Q4_K_M.gguf (1.12 GB)


In [25]:
# ── Cell 12: Push GGUF to HuggingFace ────────────────────────
from huggingface_hub import HfApi
api = HfApi()
print('Uploading GGUF (~1.2GB)...')

# Reuse the defined GGUF path and name from Cell 11
# GGUF_FULL_PATH and GGUF_FILE_NAME should be available if Cell 11 was executed successfully.
# Assuming GGUF_FULL_PATH and GGUF_FILE_NAME are defined in the previous cell and still in scope.

api.upload_file(
    path_or_fileobj=GGUF_FULL_PATH, # Use the full path for local access
    path_in_repo=GGUF_FILE_NAME,    # Use just the filename for the repo
    repo_id=HF_REPO,
    token=userdata.get('HUGGINGFACE_TOKEN'),
)
# Also upload eval results
for fname in ['baseline_results.json', 'finetuned_results.json']:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f'eval/{fname}',
        repo_id=HF_REPO,
        token=userdata.get('HUGGINGFACE_TOKEN'),
    )
print(f'All files uploaded: https://huggingface.co/{HF_REPO}')


Uploading GGUF (~1.2GB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ill-qwen-1.5b.Q4_K_M.gguf:   1%|          | 7.14MB / 1.12GB            

All files uploaded: https://huggingface.co/Nguuma/security-slm-unsloth-1.5b


In [27]:
# ── Cell 13: (Fallback) Direct download if HF upload fails ───
# Only run this cell if Cell 12 failed
from google.colab import files
files.download(GGUF_FULL_PATH) # Use the correctly defined path
files.download('baseline_results.json')
files.download('finetuned_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Cell 11b: Rename GGUF for clarity ─────────────────────────
import os

# Define the new desired name for the fine-tuned GGUF file
NEW_GGUF_FILE_NAME = f'security-slm-finetuned-{BASE_MODEL_SLUG}.Q4_K_M.gguf'
NEW_GGUF_FULL_PATH = f'{GGUF_DIR}/{NEW_GGUF_FILE_NAME}'

# Check if the original GGUF file exists before renaming
if os.path.exists(GGUF_FULL_PATH):
    os.rename(GGUF_FULL_PATH, NEW_GGUF_FULL_PATH)
    print(f'Renamed GGUF from {GGUF_FULL_PATH} to {NEW_GGUF_FULL_PATH}')

    # Update variables to reflect the new name for subsequent cells
    GGUF_FILE_NAME = NEW_GGUF_FILE_NAME
    GGUF_FULL_PATH = NEW_GGUF_FULL_PATH
else:
    print(f'Error: Original GGUF file not found at {GGUF_FULL_PATH}. Cannot rename.')


In [ ]:
# ── Install FastAPI and Uvicorn ──────────────────────────────
%pip install fastapi uvicorn

In [ ]:
# ── FastAPI Application to Consume Fine-Tuned Model ──────────

import torch
from fastapi import FastAPI
from pydantic import BaseModel
from unsloth import FastLanguageModel

# Initialize FastAPI app
app = FastAPI()

# Configuration from your notebook (adjust as needed)
HF_REPO = 'Nguuma/security-slm-unsloth-1.5b' # Your HuggingFace repo ID
MAX_SEQ_LENGTH = 2048

# Load the fine-tuned model and tokenizer
# This will load the adapter weights from HF_REPO onto the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = HF_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None, # Auto detects float16 for Tesla T4, bfloat16 for Ampere
    load_in_4bit = True,
)

# Ensure model is in evaluation mode for inference
FastLanguageModel.for_inference(model)

# Define request body for the API endpoint
class PromptRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512
    temperature: float = 0.3

# API endpoint for model inference
@app.post("/generate/")
async def generate_response(request: PromptRequest):
    formatted_prompt = f"### Instruction:\n{request.prompt}\n\n### Response:\n"
    inputs = tokenizer(formatted_prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part, similar to the notebook's run_inference
    clean_response = response.split('### Response:\n', 1)[-1].strip()

    return {"response": clean_response}

print("FastAPI app created. To run it, save this code as a Python file (e.g., app.py) and execute: uvicorn app:app --host 0.0.0.0 --port 8000")
print("Then, send POST requests to http://localhost:8000/generate/ with a JSON body like: {'prompt': 'Your question here'}")

### How to Run the FastAPI Application

1.  **Save the code:** Copy the Python code from the cell above and save it to a file named `app.py` on your local machine (or in your Colab environment if you want to run it here, but typically you'd deploy this).

2.  **Install dependencies:** If you haven't already, install FastAPI and Uvicorn:
    ```bash
    pip install fastapi uvicorn
    ```

3.  **Run the server:** Open your terminal, navigate to the directory where you saved `app.py`, and run the following command:
    ```bash
    uvicorn app:app --host 0.0.0.0 --port 8000
    ```
    If you are running this in Colab, you might need to use `nest_asyncio` and run it in a separate thread/process or use `!python -m uvicorn app:app --host 0.0.0.0 --port 8000` (though direct `uvicorn` in Colab often requires more setup for public access).

4.  **Send requests:** You can now send POST requests to `http://localhost:8000/generate/` (or the Colab public URL if you set it up) with a JSON body. For example, using `curl`:
    ```bash
    curl -X POST "http://localhost:8000/generate/" \
         -H "Content-Type: application/json" \
         -d '{ "prompt": "Explain how an attacker can craft a malicious MCP server tool description." }'
    ```

    Or using Python's `requests` library:
    ```python
    import requests

    url = "http://localhost:8000/generate/"
    headers = {"Content-Type": "application/json"}
    data = {"prompt": "Explain how an attacker can craft a malicious MCP server tool description to trick an AI agent into exfiltrating environment variables to an external server.", "max_new_tokens": 256}

    response = requests.post(url, headers=headers, json=data)
    print(response.json())
    ```

---
## Local Inference — Run on your 4GB RAM machine

### Deploy with Ollama
```bash
# Pull from HuggingFace (after Cell 12 completes)
ollama pull hf.co/your-username/security-slm-1.5b
ollama run your-username/security-slm-1.5b
```

### Or from local GGUF file
```bash
cat > Modelfile << 'EOF'
FROM ./security-slm-unsloth.Q4_K_M.gguf
SYSTEM "You are an elite AI security researcher specializing in offensive and defensive security for AI systems, cloud-native environments, and agentic AI architectures."
PARAMETER num_ctx 2048
PARAMETER temperature 0.3
EOF

ollama create security-slm -f Modelfile
ollama run security-slm
```

### Run local comparison script
```bash
# After downloading baseline_results.json and the GGUF:
python eval/compare_local.py
```

---
### Score Interpretation
| Metric | Baseline expectation | Fine-tuned target |
|--------|---------------------|-------------------|
| Avg score | 2-5/10 | 7-9/10 |
| `<think>` rate | 20-60% | 90-100% |
| Avg word count | 50-150 | 200-500 |
| Tech depth | 1-2/5 | 4-5/5 |

### Training Loss Interpretation
| Final Loss | Meaning | Action |
|-----------|---------|--------|
| > 1.5 | Underfit | Increase epochs or r=32 |
| 0.5-1.5 | Good | Proceed |
| < 0.3 | Possible overfit | Reduce epochs next run |